In [0]:
%restart_python

In [0]:
#importamos librerias

from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import countDistinct
from pyspark.ml.feature import StringIndexer
import mlflow
%pip install optuna
import optuna
spark.conf.set("spark.databricks.execution.timeout", 1800)

In [0]:
#cargamos el dataset de fraudes bancarios

df = spark.table('forecast.default.data')
df.describe().show()
df.printSchema()
df.select(countDistinct("type")).show()
df.select(countDistinct("nameOrig")).show()
df.show(10)


In [0]:

# ---------------------------
# 1. Preparación de datos
# ---------------------------

df = df.withColumn("isFraud", col("isFraud").cast("double"))

indexer = StringIndexer(
    inputCol="type",
    outputCol="type_idx",
    handleInvalid="keep"
)

numeric_features = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
]

feature_cols = numeric_features + ["type_idx"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df = indexer.fit(df).transform(df)

df_model = assembler.transform(df).select("features", "isFraud")
df_model = df_model.withColumnRenamed("isFraud", "label")

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42077651)

# ---------------------------
# 2. Evaluadores
# ---------------------------

acc_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Opcionales: precision, recall y f1
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

weighted_precision_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

weighted_recall_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

In [0]:
# ---------------------------
# 3. Experimento MLflow
# ---------------------------

mlflow.set_experiment("/Users/martoschelp@gmail.com/RandomForest_Fraud_Optuna_Spark")

# ---------------------------
# 4. Función objetivo Optuna
# ---------------------------

def objective(trial):
    params = {
        "numTrees": trial.suggest_int("numTrees", 50, 200, step=50),
        "maxDepth": trial.suggest_int("maxDepth", 3, 20),
        "maxBins": trial.suggest_int("maxBins", 32, 64, step=8),
        "minInstancesPerNode": trial.suggest_int("minInstancesPerNode", 1, 5)
    }

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        seed=42,
        **params
    )

    model = rf.fit(train_df)
    preds = model.transform(test_df)

    acc = acc_eval.evaluate(preds)
    auc = auc_eval.evaluate(preds)
    f1 = f1_eval.evaluate(preds)
    weighted_precision = weighted_precision_eval.evaluate(preds)
    weighted_recall = weighted_recall_eval.evaluate(preds)

    # Logging del trial en MLflow
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("weighted_precision", weighted_precision)
        mlflow.log_metric("weighted_recall", weighted_recall)

    # Guardar info útil en Optuna
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("weighted_precision", weighted_precision)
    trial.set_user_attr("weighted_recall", weighted_recall)

    return auc

# ---------------------------
# 5. Optimización
# ---------------------------

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

# ---------------------------
# 6. Ver mejores trials
# ---------------------------

top_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:3]

for trial in top_trials:
    print(f"Trial #{trial.number}")
    print("Params:", trial.params)
    print("AUC:", trial.value)
    print("Accuracy:", trial.user_attrs["accuracy"])
    print("F1:", trial.user_attrs["f1"])
    print("Weighted Precision:", trial.user_attrs["weighted_precision"])
    print("Weighted Recall:", trial.user_attrs["weighted_recall"])
    print("-" * 50)

In [0]:
import optuna.visualization

# Visualización de la optimización con Optuna
optuna.visualization.plot_optimization_history(study)
optuna.visualization.plot_param_importances(study)
optuna.visualization.plot_parallel_coordinate(study)
optuna.visualization.plot_slice(study)